# 00 · Ortam kontrolü

Kurulumun son adımı (docs/10-kurulum-rehberi.md, bölüm 5): Python sürümünü, API anahtarını, `data/` yollarını, EVREN bağlantısını ve birim testlerini doğrular. Sonuçları `docs/10-kurulum-rehberi.md` içine not et.


In [ ]:
# Kurulum hücresi: bütün notebook'larda aynı (bkz. notebooks/README.md)
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
%run {REPO}/notebooks/_kurulum.py


## Veri yolları


In [ ]:
from ruya.config import ayarlar

ayar = ayarlar()
for ad in (
    "ham_klasoru",
    "islenmis_klasoru",
    "llm_onbellek_klasoru",
    "rapor_klasoru",
    "chroma_klasoru",
):
    yol = getattr(ayar, ad)
    print(f"{ad:22} {yol}  {'var' if yol.exists() else 'yok'}")


## EVREN bağlantısı

Tek kısa istek gönderilir; servis yoğunsa bir dakikadan uzun sürebilir. Yanıt önbelleğe girer, ikinci çalıştırma anında döner.


In [ ]:
import time

from ruya import config
from ruya.llm.client import LLMIstemcisi

istek = {
    "model": config.MODEL_CIKARIM,
    "max_tokens": 1000,
    "messages": [{"role": "user", "content": 'Yalnızca "tamam" yaz.'}],
}
baslangic = time.perf_counter()
yanit = LLMIstemcisi().mesaj(istek)
sure = time.perf_counter() - baslangic
print(f"Süre     : {sure:.2f} sn")
print(f"Yanıt    : {yanit.icerik}")
print(f"Kullanım : {yanit.kullanim}")


## Birim testleri

Verisiz testler her zaman çalışır; veri indirildiyse kontrat testleri de çalışır (`-m veri`).


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO, check=False)
